In [2]:
import json
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re

In [4]:
URL = "https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html"

html = requests.get(URL).text
soup = BeautifulSoup(html, "html.parser")

print("Título:", soup.title.text)

print("\nPrimeros enlaces:\n")

for i, a in enumerate(soup.find_all("a", href=True)[:40]):
    texto = a.get_text(" ", strip=True)
    href = a["href"]

    if texto:
        print(f"{i:2d} | {texto[:80]}")
        print("   ", href)

Título: Formación online UPV |CFP

Primeros enlaces:

 0 | Registrarse
    https://poseidon.cfp.upv.es/portal-formacion/registro/registro.jsp?idioma=es
 1 | Buscar formación
    /formacion-permanente/inicio/buscador_cursos.html
 2 | Contacto
    /formacion-permanente/general/contacto.html
 4 | ES
    https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html?idioma=es
 5 | VA
    https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html?idioma=va
 6 | ES
    https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html?idioma=es
 7 | VA
    https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html?idioma=va
 9 | Iniciar sesión
    https://poseidon.cfp.upv.es/portal-formacion/registro/identif.jsp?idioma=es
10 | Inicio
    /formacion-permanente
11 | Másteres y diplomas
    javascript:void(0);
12 | Arquitectura, ingeniería civil y edificación
    /formacion-permanente/masters/masters-area.html?areaInteres=CO
13 | Arte y Humanidades
  

In [5]:
from collections import Counter

counter = Counter()

for tag in soup.find_all():
    counter[tag.name] += 1

print(counter.most_common(25))

[('li', 944), ('i', 399), ('div', 366), ('a', 219), ('script', 173), ('ul', 167), ('span', 158), ('h3', 158), ('meta', 20), ('link', 16), ('p', 5), ('img', 4), ('button', 3), ('input', 2), ('html', 1), ('head', 1), ('title', 1), ('body', 1), ('h2', 1), ('br', 1), ('form', 1), ('h4', 1)]


In [6]:
for t in soup.find_all(string=lambda s: s and "Curso" in s):
    print("="*80)
    print(t.parent.prettify()[:3000])
    break

<a class="dropdown-toggle" data-toggle="dropdown" href="javascript:void(0);">
 Cursos y jornadas
</a>



In [7]:
# ─────────────────────────────────────────────
# EXPLORADOR FORMACIÓN PERMANENTE UPV
# ─────────────────────────────────────────────

!pip install -q beautifulsoup4 requests

import requests
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin


BASE_URL = "https://www.cfp.upv.es"


URLS = {
    "cursos_online":
        "https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html",

    "masters":
        "https://www.cfp.upv.es/formacion-permanente/masters/masters.html"
}


HEADERS = {
    "User-Agent":
    "Mozilla/5.0 (compatible; UPV-KB-Bot/1.0)"
}


def descargar(url):
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


for origen, url in URLS.items():

    print("\n" + "="*80)
    print("ORIGEN:", origen)
    print(url)
    print("="*80)

    soup = descargar(url)


    enlaces = []

    for a in soup.find_all("a", href=True):

        texto = a.get_text(" ", strip=True)
        href = a["href"]

        if not texto:
            continue

        texto_norm = texto.lower()

        # filtros candidatos
        if (
            "curso" in texto_norm
            or "máster" in texto_norm
            or "master" in texto_norm
            or "diploma" in texto_norm
            or "ects" in texto_norm
        ):
            enlaces.append(
                (
                    texto,
                    urljoin(BASE_URL, href)
                )
            )


    print(
        f"\nEnlaces candidatos encontrados: {len(enlaces)}"
    )


    for i, (texto, href) in enumerate(enlaces[:30]):

        print(
            f"\n{i:02d} | {texto}"
        )

        print(
            "   ",
            href
        )


    # estadísticas tags
    print("\nTags principales:")

    contador = {}

    for tag in soup.find_all():
        contador[tag.name] = contador.get(tag.name, 0) + 1

    for k,v in sorted(
        contador.items(),
        key=lambda x:x[1],
        reverse=True
    )[:15]:
        print(k, v)


    # buscar textos con ECTS / Curso
    print("\nFragmentos con ECTS o Curso:")

    textos = soup.get_text("\n", strip=True).split("\n")

    encontrados = []

    for t in textos:
        if (
            re.search(
                r"(ECTS|Curso|Máster|Master|Diploma)",
                t,
                re.I
            )
        ):
            encontrados.append(t)


    for t in encontrados[:20]:
        print("-", t)


ORIGEN: cursos_online
https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html

Enlaces candidatos encontrados: 7

00 | Másteres y diplomas
    javascript:void(0);

01 | Catálogo de Másteres
    https://www.cfp.upv.es/formacion-permanente/masters/masters.html

02 | Másteres y otros diplomas online
    https://www.cfp.upv.es/formacion-permanente/masters/masters-online.html

03 | Cursos y jornadas
    javascript:void(0);

04 | Cursos online
    https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html

05 | Cursos para los próximos 30 días
    https://www.cfp.upv.es/formacion-permanente/cursos-y-jornadas/cursos-y-jornadas.html

06 | GESTIÓN DE RECURSOS HUMANOS
    https://www.cfp.upv.es/formacion-permanente/curso/gestion-recursos-humanos_104137.html

Tags principales:
li 944
i 399
div 366
a 219
script 173
ul 167
span 158
h3 158
meta 20
link 16
p 5
img 4
button 3
input 2
html 1

Fragmentos con ECTS o Curso:
- Másteres y diplomas
- Másteres y diplomas: áreas

In [8]:
# ─────────────────────────────────────────────
# GENERADOR JSON FORMACIÓN PERMANENTE UPV
# ─────────────────────────────────────────────

!pip install -q beautifulsoup4 requests

import os
import re
import json
import time
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


BASE_URL = "https://www.cfp.upv.es"


FUENTES = [
    (
        "cursos_online",
        "https://www.cfp.upv.es/formacion-permanente/online/formacion-online.html"
    ),
    (
        "masters",
        "https://www.cfp.upv.es/formacion-permanente/masters/masters.html"
    )
]


SALIDA = "/content/drive/MyDrive/TFG Teleco/JSONs/formacion_permanente_upv.json"


HEADERS = {
    "User-Agent":
    "Mozilla/5.0 (compatible; UPV-KB-Bot/1.0)"
}


PAUSA = 0.3



def get(url):

    try:
        r = requests.get(
            url,
            headers=HEADERS,
            timeout=20
        )

        r.raise_for_status()
        time.sleep(PAUSA)

        return BeautifulSoup(
            r.text,
            "html.parser"
        )

    except Exception as e:
        print("⚠️ error:", url, e)
        return None



def limpiar_nombre(t):

    t = re.sub(
        r"\s+",
        " ",
        t
    ).strip()

    return t



def clasificar_tipo(nombre):

    n = nombre.lower()

    if "máster" in n or "master" in n:
        return "Master"

    if "diploma de especialización" in n:
        return "Diploma de especialización"

    if "diploma de experto" in n:
        return "Diploma de experto"

    if "diploma de extensión" in n:
        return "Diploma de extensión"

    if "curso" in n:
        return "Curso"

    return "Otro"



def extraer_fichas(origen, url):

    soup = get(url)

    if not soup:
        return []


    encontrados = {}

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if "/formacion-permanente/curso/" not in href:
            continue


        url_final = urljoin(
            BASE_URL,
            href
        )


        nombre = limpiar_nombre(
            a.get_text(" ", strip=True)
        )

##############################################################
        # eliminar enlaces basura
        basura = [
            "Matriculable",
            "Más información",
            "Ver más",
            "Acceder",
            "Inscribirse"
        ]

        if any(
            b.lower() == nombre.lower()
            for b in basura
        ):
            continue
      ############################################################

        if len(nombre) < 5:
            continue


        encontrados[url_final] = {
            "nombre": nombre,
            "url": url_final,
            "origen": origen
        }


    return list(encontrados.values())



def extraer_detalle(item):

    soup = get(item["url"])

    if not soup:
        return item


    texto = soup.get_text(
        "\n",
        strip=True
    )


    # tipo
    item["tipo"] = clasificar_tipo(
        item["nombre"]
    )


    # ECTS
    m = re.search(
        r"(\d+(?:,\d+)?)\s*ECTS",
        texto,
        re.I
    )

    item["ects"] = (
        m.group(1)
        if m
        else None
    )


    # horas
    m = re.search(
        r"(\d+)\s*horas",
        texto,
        re.I
    )

    item["horas"] = (
        m.group(1)
        if m
        else None
    )


    # precio
    m = re.search(
        r"(\d+[.,]?\d*)\s*€",
        texto
    )

    item["precio"] = (
        m.group(1) + " €"
        if m
        else None
    )


    # campus
    campus = [
        "Vera",
        "Alcoy",
        "Gandia"
    ]

    item["campus"] = None

    for c in campus:
        if c.lower() in texto.lower():
            item["campus"] = c
            break


    # responsable/promotor
    claves = [
        "Promueve",
        "Organiza",
        "Responsable",
        "Director"
    ]

    item["promotor"] = None
    item["responsable"] = None


    for linea in texto.split("\n"):

        for clave in claves:

            if linea.lower().startswith(
                clave.lower()
            ):

                valor = linea.split(
                    ":",
                    1
                )

                if len(valor)==2:

                    if clave in ["Promueve","Organiza"]:
                        item["promotor"] = valor[1].strip()

                    else:
                        item["responsable"] = valor[1].strip()



    return item




# ─────────────────────────────────────────────
# EJECUCIÓN
# ─────────────────────────────────────────────


catalogo = {}



for origen, url in FUENTES:

    print("\nProcesando:", origen)

    fichas = extraer_fichas(
        origen,
        url
    )

    print(
        "Encontradas:",
        len(fichas)
    )


    for f in fichas:

        # deduplicación por URL
        catalogo[f["url"]] = f



print(
    "\nTotal sin duplicados:",
    len(catalogo)
)



resultado = []


for i,item in enumerate(
    catalogo.values(),
    1
):

    print(
        f"[{i}/{len(catalogo)}]",
        item["nombre"]
    )

    item = extraer_detalle(item)


    # id para nombre de archivo posterior
    item["id"] = re.sub(
        r"[^a-z0-9]+",
        "_",
        item["nombre"].lower()
    ).strip("_")


    resultado.append(item)



datos_finales = {
    "fuente": "Formación Permanente UPV",
    "total": len(resultado),
    "formaciones": resultado
}



os.makedirs(
    os.path.dirname(SALIDA),
    exist_ok=True
)


with open(
    SALIDA,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        datos_finales,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\n✅ JSON generado:")
print(SALIDA)
print(
    "Elementos:",
    len(resultado)
)


Procesando: cursos_online
Encontradas: 152

Procesando: masters
Encontradas: 136

Total sin duplicados: 288
[1/288] ANÁLISIS DE LA COYUNTURA ECONÓMICA
[2/288] MODELOS MULTICRITERIO APLICADOS A LA GESTIÓN DE CARTERAS
[3/288] GESTIÓN DE CARTERAS II
[4/288] INCENDIOS DE ORIGEN ELÉCTRICO EN EL ÁMBITO DOMÉSTICO. CAUSAS, RIESGOS Y ACTUACIÓN
[5/288] CAMPOS MAGNÉTICOS EN INSTALACIONES ELÉCTRICAS Y ALREDEDORES, Y SU CÁLCULO Y REPRESENTACIÓN CON CRMAG PLUS
[6/288] ANÁLISIS Y DISEÑO DE PUESTAS A TIERRA EN INSTALACIONES ELÉCTRICAS CON CRGROUND®
[7/288] ASESOR FINANCIERO
[8/288] AGENTE FINANCIERO EUROPEO
[9/288] ASISTENTE FINANCIERO EUROPEO
[10/288] ACTUALIZACIÓN DE CONOCIMIENTOS EN ASESORÍA FINANCIERA 2025
[11/288] ACTUALIZACIÓN DE CONOCIMIENTOS EN CRÉDITO INMOBILIARIO 2025
[12/288] ASESOR FINANCIERO EN CRÉDITO HIPOTECARIO
[13/288] INFORMADOR FINANCIERO EN CRÉDITO HIPOTECARIO
[14/288] CLOUD COMPUTING CON AMAZON WEB SERVICES (AWS)
[15/288] BASES DE DATOS ESPACIALES: POSTGIS
[16/288] INTRODUCCIÓN A

In [18]:
import os
import shutil

# Buscar cualquier carpeta JSONs dentro del Drive
carpeta_json = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if os.path.basename(root) == "JSONs":
        carpeta_json = root
        break

if carpeta_json is None:
    raise Exception("No se ha encontrado ninguna carpeta JSONs")


print("Carpeta encontrada:")
print(carpeta_json)


# Buscar el JSON generado de formación permanente
origen = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "formacion_permanente_upv.json" in files:
        origen = os.path.join(root, "formacion_permanente_upv.json")
        break


if origen is None:
    raise Exception("No se ha encontrado formacion_permanente_upv.json")


print("\nOrigen:")
print(origen)


# Copiar al JSONs correcto
destino = os.path.join(
    carpeta_json,
    "formacion_permanente_upv.json"
)

shutil.copy2(origen, destino)


print("\nGuardado correctamente:")
print(destino)


# Mostrar contenido final de la carpeta
print("\nContenido de JSONs:")
for f in os.listdir(carpeta_json):
    print("-", f)

Carpeta encontrada:
/content/drive/MyDrive/TFG Teleco/JSONs

Origen:
/content/drive/MyDrive/TFG Teleco/JSONs/formacion_permanente_upv.json


SameFileError: '/content/drive/MyDrive/TFG Teleco/JSONs/formacion_permanente_upv.json' and '/content/drive/MyDrive/TFG Teleco/JSONs/formacion_permanente_upv.json' are the same file

In [26]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "doctorados_upv.json" in files:
        print("Encontrado:")
        print(os.path.join(root, "doctorados_upv.json"))
